# Unidad 02 - Transacciones

In [ ]:
USE Pampero;
-- Comienzo una nueva transaccion
BEGIN TRAN;
-- Declaro una variable
DECLARE @nuevoordenid AS INT;
-- Inserto un nuevo pedido en la tabla Pedidos
INSERT INTO Pedidos
(IDCliente, IDEmpleado, FechaPedido, FechaRequerida, FechaEnvio, EnvioPor, Flete, NombreEnvio, DireccionEnvio, CiudadEnvio, CodigoPostalEnvio, PaisEnvio)
VALUES
('XXXXX', 5, '20220212', '20220301', '20220216',
3, 32.38, 'Ship to 85-B', '6789 rue de l''Abbaye', 'Reims', '10345', 'Francia');
-- Guardo el nuevo ID de orden en una variable
SET @nuevoordenid = SCOPE_IDENTITY();
-- Devuelvo el nuevo ID de orden
SELECT @nuevoordenid AS nuevoordenid;
-- Inserto lineas de pedido para el nuevo pedido 
INSERT INTO [Detalles Pedido]
(IDPedido, IDProducto, PrecioUnitario, Cantidad, Descuento)
VALUES(@nuevoordenid, 11, 14.00, 12, 0.000),
(@nuevoordenid, 42, 9.80, 10, 0.000),
(@nuevoordenid, 72, 34.80, 5, 0.000);
-- Confirmo la transaccion
COMMIT TRAN

## Trabas y bloqueos
### Conexión 01

In [ ]:
USE Pampero;
BEGIN TRAN;
UPDATE Productos
SET PrecioUnitario += 1.00
WHERE IDProducto = 2;

### Conexión 02

In [ ]:
USE Pampero;
SELECT IDProducto, PrecioUnitario
FROM Productos
WHERE IDProducto = 2;

### Para definir un timeout (en milisegundos)

In [ ]:
SET LOCK_TIMEOUT 5000;
SELECT IDProducto, PrecioUnitario
FROM Productos
WHERE IDProducto = 2;

### Para volver a dejar timeout indefinido

In [ ]:
SET LOCK_TIMEOUT -1;
SELECT IDProducto, PrecioUnitario
FROM Productos
WHERE IDProducto = 2;

## Interbloqueos (DeadLocks)
### Conexión 01

In [ ]:
USE Pampero;
BEGIN TRAN;
UPDATE Productos
SET PrecioUnitario += 1.00
WHERE IDProducto = 2;

### Conexión 02

In [ ]:
USE Pampero;
BEGIN TRAN;
UPDATE [Detalles Pedido]
SET PrecioUnitario += 1.00
WHERE IDProducto = 2;

### Conexión 01

In [ ]:
SELECT IDPedido, IDProducto, PrecioUnitario
FROM [Detalles Pedido]
WHERE IDProducto = 2;
COMMIT TRAN;

### Conexión 02

In [ ]:
SELECT IDProducto, PrecioUnitario
FROM Productos
WHERE IDProducto = 2;
COMMIT TRAN;

## Tablas temporales

Ejemplo:

|Año|Cantidad|
|:----|--------:|
|2019 | 9581|
|2020 | 25489|
|2021 | 16247|

Pero necesitamos:
|Año|Cantidad|Cantidad Año Previo|
|:----|--------:|--------------:|
|2019 | 9581||
|2020 | 25489|9581|
|2021 | 16247|25489|


In [9]:
DROP TABLE IF EXISTS #MiTotalOrdenesPorAnio;
GO
CREATE TABLE #MiTotalOrdenesPorAnio (
ordenanio INT NOT NULL PRIMARY KEY, 
cantidad INT NOT NULL
);

INSERT INTO #MiTotalOrdenesPorAnio(ordenanio, cantidad) 
SELECT YEAR(O.FechaPedido) AS ordenanio, SUM(Do.Cantidad) AS cantidad
FROM Pedidos AS O
INNER JOIN [Detalles Pedido] AS DO ON DO.IDPedido = O.IDPedido
GROUP BY YEAR(FechaPedido);

SELECT Act.ordenanio, Act.cantidad AS cantidadanioact, Prv.cantidad AS cantidadanioprv 
FROM #MiTotalOrdenesPorAnio AS Act
LEFT OUTER JOIN #MiTotalOrdenesPorAnio AS Prv ON Act.ordenanio = Prv.ordenanio + 1;

Commands completed successfully.

(3 rows affected)
(3 rows affected)

ordenanio | cantidadanioact | cantidadanioprv
----------+-----------------+----------------
2019      | 9581            | NULL           
2020      | 25489           | 9581           
2021      | 16247           | 25489          
(3 rows)

Total execution time: 00:00:17.918

### Variables de tabla

In [10]:
DECLARE @MiTotalOrdenesPorAnio TABLE (
anioorden INT NOT NULL PRIMARY KEY, 
cantidad INT NOT NULL);

INSERT INTO @MiTotalOrdenesPorAnio(anioorden, cantidad)
SELECT YEAR(O.FechaPedido) AS ordenanio, SUM(Do.cantidad) AS cantidad
FROM Pedidos AS O
INNER JOIN [Detalles Pedido] AS DO ON DO.IDPedido = O.IDPedido
GROUP BY YEAR(FechaPedido);

SELECT Act.anioorden, Act.cantidad AS actaniocant, Prv.cantidad AS prvaniocant
FROM @MiTotalOrdenesPorAnio AS Act
LEFT OUTER JOIN @MiTotalOrdenesPorAnio AS Prv 
ON Act.anioorden = Prv.anioorden + 1;

(3 rows affected)
(3 rows affected)

anioorden | actaniocant | prvaniocant
----------+-------------+------------
2019      | 9581        | NULL       
2020      | 25489       | 9581       
2021      | 16247       | 25489      
(3 rows)

Total execution time: 00:00:00.022

### Con Funciones de Ventana (las veremos en la próxima unidad)

In [11]:
SELECT YEAR(O.FechaPedido) AS anioorden, 
SUM(DO.cantidad) AS actaniocant,
LAG(SUM(DO.cantidad)) OVER(ORDER BY YEAR(FechaPedido)) AS prvaniocant 
FROM Pedidos AS O
INNER JOIN [Detalles Pedido] AS DO ON DO.IDPedido = O.IDPedido
GROUP BY YEAR(FechaPedido);

(3 rows affected)

anioorden | actaniocant | prvaniocant
----------+-------------+------------
2019      | 9581        | NULL       
2020      | 25489       | 9581       
2021      | 16247       | 25489      
(3 rows)

Total execution time: 00:00:00.009